# Titanic — анализ датасета

 Данные — `/content/dataset/train.csv`

| # | Задача | Функция |
|---|---|---|
| 2 | Обзор датасета | `overview.run(df)` |
| 3 | Выживаемость по классам | `survival.run(df)` |
| 4 | Популярные имена | `names.run(df)` |
| 5 | Популярные имена по классам | `names.run_by_class(df)` |
| 6 | Пассажиры старше 44 | `filters.run_older(df)` |
| 7 | Мужчины младше 44 | `filters.run_younger_males(df)` |
| 8 | n-местные каюты | `cabins.run(df)` |

## 1. Данные

Приложение ожидает расположение файла данных для анализа по пути  — `/content/Dataset/train.csv`.

In [20]:
# ── загрузка данных: /content/dataset/train.csv
from pathlib import Path
import pandas as pd

DATASET_PATH = Path('/content/Dataset/train.csv')

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Не найден {DATASET_PATH}. ')

df = pd.read_csv(DATASET_PATH, index_col='PassengerId')
print(f'загружено: {df.shape[0]} строк × {df.shape[1]} столбцов')
df.head()

загружено: 891 строк × 11 столбцов


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Утилиты

Функции, нужные более чем одной задаче. Без этой ячейки `names` и `overview`
не заработают.

In [21]:
# ── утилиты: извлечение имени и отчёт о пропусках
import re
import pandas as pd

# У замужних женщин собственное имя записано в скобках.
PARENTHESES = re.compile(r"\(([^)]*)\)")
# Прозвище в кавычках ("Annie") в имя не берём.
QUOTED = re.compile(r'"[^"]*"')


def extract_first_name(full_name: str) -> str:
    """Личное имя из строки «Фамилия, Титул. Имя Отчество»."""
    rest = full_name.split('.', 1)[1] if '.' in full_name else full_name
    match = PARENTHESES.search(rest)
    if match:
        inside = match.group(1).strip()
        if not inside.startswith('"'):
            rest = inside
    rest = QUOTED.sub('', rest)
    rest = PARENTHESES.sub('', rest)
    words = rest.split()
    return words[0] if words else ''


def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    """Количество и процент пропусков по каждому столбцу."""
    count = df.isna().sum()
    return pd.DataFrame({
        'пропусков': count,
        'процент': (count / len(df) * 100).round(2),
    })

## 3. Задачи

### Задача 2. Обзор датасета

In [ ]:
# ── задача 2: типы столбцов, пропуски, описательная статистика
import pandas as pd


def overview_run(df: pd.DataFrame) -> None:
    print('--- Типы данных и заполненность ---')
    df.info()
    print('\n--- Пропуски ---')
    print(missing_report(df))
    print('\n--- Описательная статистика ---')
    print(df.describe().round(2))


overview_run(df)

--- Типы данных и заполненность ---
<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    object 
 3   Sex       891 non-null    object 
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    object 
 8   Fare      891 non-null    float64
 9   Cabin     204 non-null    object 
 10  Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 83.5+ KB

--- Пропуски ---
          пропусков  процент
Survived          0     0.00
Pclass            0     0.00
Name              0     0.00
Sex               0     0.00
Age             177    19.87
SibSp             0     0.00
Parch             0     0.00
Ticket            0     0.00
Fare           

### Задача 3. Выживаемость по классам

In [ ]:
# ── задача 3: доля выживших по классам (в процентах)
import pandas as pd


def survival_run(df: pd.DataFrame) -> None:
    rate = (df.groupby('Pclass')['Survived'].mean() * 100).round(2)
    print('--- Процент выживших по классам ---')
    print(rate)


survival_run(df)

--- Процент выживших по классам ---
Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64


### Задача 4. Популярные имена на корабле

In [ ]:
# ── задача 4: самое частое имя среди мужчин и среди женщин
import pandas as pd


def most_popular(df: pd.DataFrame, sex: str) -> tuple[str, int]:
    counts = df[df['Sex'] == sex]['Name'].apply(extract_first_name).value_counts()
    return counts.index[0], int(counts.iloc[0])


def names_run(df: pd.DataFrame) -> None:
    print('--- Самые популярные имена на корабле ---')
    for sex, label in (('male', 'мужское'), ('female', 'женское')):
        name, count = most_popular(df, sex)
        print(f'{label}: {name} ({count} чел.)')


names_run(df)

--- Самые популярные имена на корабле ---
мужское: William (35 чел.)
женское: Anna (15 чел.)


### Задача 5. Популярные имена по классам

In [ ]:
# ── задача 5: таблица «класс × пол → самое частое имя»
import pandas as pd


def names_by_class_run(df: pd.DataFrame) -> None:
    rows = []
    for pclass in sorted(df['Pclass'].unique()):
        in_class = df[df['Pclass'] == pclass]
        for sex, label in (('male', 'мужское'), ('female', 'женское')):
            name, count = most_popular(in_class, sex)
            rows.append({'Pclass': pclass, 'пол': label, 'имя': name, 'человек': count})
    table = pd.DataFrame(rows).set_index(['Pclass', 'пол'])
    print('--- Самые популярные имена в каждом классе ---')
    print(table)


names_by_class_run(df)

--- Самые популярные имена в каждом классе ---
                      имя  человек
Pclass пол                        
1      мужское    William       11
       женское  Elizabeth        5
2      мужское    William        9
       женское  Elizabeth        5
3      мужское    William       15
       женское       Anna        9


### Задача 6. Пассажиры старше 44 лет

In [ ]:
# ── задача 6: строки с Age > 44
import pandas as pd


def older_run(df: pd.DataFrame, age: float = 44) -> None:
    subset = df[df['Age'] > age]
    print(f'--- Пассажиры старше {age} лет: {len(subset)} чел. ---')
    print(subset)


older_run(df)

--- Пассажиры старше 44 лет: 115 чел. ---
             Survived  Pclass  \
PassengerId                     
7                   0       1   
12                  1       1   
16                  1       2   
34                  0       2   
53                  1       1   
...               ...     ...   
858                 1       1   
863                 1       1   
872                 1       1   
874                 0       3   
880                 1       1   

                                                          Name     Sex   Age  \
PassengerId                                                                    
7                                      McCarthy, Mr. Timothy J    male  54.0   
12                                    Bonnell, Miss. Elizabeth  female  58.0   
16                            Hewlett, Mrs. (Mary D Kingcome)   female  55.0   
34                                       Wheadon, Mr. Edward H    male  66.0   
53                    Harper, Mrs. Henry Sleeper

### Задача 7. Мужчины младше 44 лет

In [ ]:
# ── задача 7: мужчины с возрастом в (0, 44)
# Age > 0 дополнительно отсекает NaN: сравнение NaN с числом даёт False
import pandas as pd


def younger_males_run(df: pd.DataFrame, age: float = 44) -> None:
    subset = df[(df['Age'] > 0) & (df['Age'] < age) & (df['Sex'] == 'male')]
    print(f'--- Мужчины младше {age} лет: {len(subset)} чел. ---')
    print(subset)


younger_males_run(df)

--- Мужчины младше 44 лет: 368 чел. ---
             Survived  Pclass                            Name   Sex   Age  \
PassengerId                                                                 
1                   0       3         Braund, Mr. Owen Harris  male  22.0   
5                   0       3        Allen, Mr. William Henry  male  35.0   
8                   0       3  Palsson, Master. Gosta Leonard  male   2.0   
13                  0       3  Saundercock, Mr. William Henry  male  20.0   
14                  0       3     Andersson, Mr. Anders Johan  male  39.0   
...               ...     ...                             ...   ...   ...   
884                 0       2   Banfield, Mr. Frederick James  male  28.0   
885                 0       3          Sutehall, Mr. Henry Jr  male  25.0   
887                 0       2           Montvila, Rev. Juozas  male  27.0   
890                 1       1           Behr, Mr. Karl Howell  male  26.0   
891                 0       3       

### Задача 8. n-местные каюты

In [ ]:
# ── задача 8: сколько кают на 2, 3, 4, ... человека
import pandas as pd


def cabins_run(df: pd.DataFrame, min_people: int = 2) -> None:
    sizes = df['Cabin'].dropna().value_counts().value_counts().sort_index()
    sizes = sizes[sizes.index >= min_people]
    sizes.index.name = 'человек в каюте'
    sizes.name = 'количество кают'
    print('--- Количество n-местных кают ---')
    print(sizes)


cabins_run(df)

--- Количество n-местных кают ---
человек в каюте
2    38
3     5
4     3
Name: количество кают, dtype: int64
